# Установка
- [Spark UI](http://clientnode:4040/)

In [ ]:
!pip install pyspark

In [1]:
import pyspark.sql as pyspark
import pyspark.sql.functions as F

In [2]:
%%html
<style>
.jp-OutputArea-output pre {
    white-space: pre;
}
</style>

### Создание сессии

In [3]:
spark = (pyspark.SparkSession.builder
    .master('local')
    .appName("My Spark Application")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark

# Датасет IMDb Non-Commercial Datasets
ссылка на скачивание и описание
https://developer.imdb.com/non-commercial-datasets/

# Чтение данных

In [ ]:
name_basics_df = ...

In [20]:
title_akas_df = spark.read.csv('title.akas.tsv.gz', sep='\t', header=True, nullValue=r'\N')

In [23]:
title_basics_df = ...

In [24]:
title_crew_df = ...

In [25]:
title_principals_df = ...

In [26]:
ratings_df = ...

In [27]:
name_basics_df = ...

# Изучить данные

In [34]:
title_akas_df.show()

+---------+--------+--------------------+------+--------+-----------+--------------------+---------------+
|  titleId|ordering|               title|region|language|      types|          attributes|isOriginalTitle|
+---------+--------+--------------------+------+--------+-----------+--------------------+---------------+
|tt0000004|       1|         Un bon bock|  NULL|    NULL|   original|                NULL|              1|
|tt0000004|       2|         A Good Beer|  NULL|    NULL|       NULL|                NULL|              0|
|tt0000004|       7|  Полная кружка пива|    RU|    NULL|imdbDisplay|                NULL|              0|
|tt0000034|       1|Arrivée d'un trai...|  NULL|    NULL|   original|                NULL|              1|
|tt0000034|       5|Прибытие поезда н...|    RU|    NULL|imdbDisplay|                NULL|              0|
|tt0000035|       1|          L'arroseur|  NULL|    NULL|   original|                NULL|              1|
|tt0000035|       4|Watering the Flow

In [43]:
ratings_df.show(10)

+-------------+--------+---------+
|averageRating|numVotes|   tconst|
+-------------+--------+---------+
|          5.7|    2193|tt0000001|
|          6.8|    8240|tt0000010|
|          6.2|    1315|tt0000015|
|          5.1|    1244|tt0000022|
|          5.7|    1637|tt0000023|
|          5.5|    1129|tt0000031|
|          4.4|     686|tt0000036|
|          4.1|      87|tt0000040|
|          3.8|      42|tt0000045|
|          4.5|      66|tt0000049|
+-------------+--------+---------+
only showing top 10 rows


In [39]:
name_basics_df.show()

+----------+--------------------+---------+---------+--------------------+--------------------+
|    nconst|         primaryName|birthYear|deathYear|   primaryProfession|      knownForTitles|
+----------+--------------------+---------+---------+--------------------+--------------------+
|nm14679231|     Mussarat Sheikh|       \N|       \N|               actor|           tt7511032|
|nm16079858|          Yaco Lorca|       \N|       \N|               actor|           tt0070250|
|nm17345477|         Niana Renee|       \N|       \N|             actress|tt37520663,tt3704...|
| nm0118320|         Andrea Buck|       \N|       \N|producer,producti...|tt0105298,tt07911...|
| nm8695052|       Alex Martinez|       \N|       \N|                  \N|                  \N|
|nm17042966|      Porter Stanley|       \N|       \N|       miscellaneous|          tt10483062|
| nm4682623|           Chris Hui|       \N|       \N|      visual_effects|           tt0458290|
| nm1758290|         Tom Chaplin|     19

# Фильтрация

## Найти властелин колец

In [50]:
title_basics_df.filter(F.col('tconst') == "tt0120737").show()

[Stage 40:===========================================>              (3 + 0) / 4]

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|tt0120737|    movie|The Lord of the R...|The Lord of the R...|      0|     2001|   NULL|           178|Adventure,Drama,F...|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+



## Найти создателей в crew (split + isin)

In [79]:
writers = title_crew_df.filter('`tconst` == "tt0120737"').select(F.split('writers', ',').alias('writers')).limit(1000).collect()

In [76]:
writers = writers[0].writers

In [78]:
name_basics_df.filter(F.col('nconst').isin(writers)).show()

[Stage 55:>                                                         (0 + 1) / 1]

+---------+---------------+---------+---------+--------------------+--------------------+
|   nconst|    primaryName|birthYear|deathYear|   primaryProfession|      knownForTitles|
+---------+---------------+---------+---------+--------------------+--------------------+
|nm0001392|  Peter Jackson|     1961|       \N|producer,director...|tt0120737,tt00926...|
|nm0866058| J.R.R. Tolkien|     1892|     1973|writer,director,p...|tt0120737,tt01672...|
|nm0909638|     Fran Walsh|     1959|       \N|writer,casting_de...|tt0120737,tt01672...|
|nm0101991|Philippa Boyens|       \N|       \N|writer,producer,a...|tt0120737,tt01672...|
+---------+---------------+---------+---------+--------------------+--------------------+



# Сортировка

## Найти самый длиный фильм (cast)

In [82]:
title_basics_df.printSchema()

root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)



In [106]:
longest_films = title_basics_df.sort(F.col('runtimeMinutes').cast('int') / 60, ascending=False).filter("`titleType` == 'movie'").limit(10).collect()

# Join

## Вывести фильмы с самым высоким рейтингом

In [107]:
title_crew_df.show()

+---------+-------------------+---------+
|   tconst|          directors|  writers|
+---------+-------------------+---------+
|tt0000001|          nm0005690|     NULL|
|tt0000010|          nm0525910|     NULL|
|tt0000015|          nm0721526|nm0721526|
|tt0000022|          nm0525910|     NULL|
|tt0000023|          nm0525910|     NULL|
|tt0000031|          nm0525910|     NULL|
|tt0000036|          nm0005690|nm0410331|
|tt0000040|          nm0617588|     NULL|
|tt0000045|          nm0617588|     NULL|
|tt0000049|          nm0010291|     NULL|
|tt0000053|          nm0684607|     NULL|
|tt0000057|          nm0617588|     NULL|
|tt0000082|          nm0005690|     NULL|
|tt0000083|          nm0617588|     NULL|
|tt0000089|nm0525908,nm0698645|     NULL|
|tt0000091|          nm0617588|nm0617588|
|tt0000095|          nm0617588|     NULL|
|tt0000096|          nm0617588|     NULL|
|tt0000097|          nm0617588|     NULL|
|tt0000103|          nm0617588|     NULL|
+---------+-------------------+---

In [120]:
big_df = (title_basics_df
    .join(ratings_df, 'tconst')
    .join(title_crew_df, title_basics_df.tconst == title_crew_df.tconst)
    .join(name_basics_df, F.col('directors') == name_basics_df.nconst)
)

In [121]:
big_df.count()

1113976

In [119]:
big_df.cache()

DataFrame[tconst: string, titleType: string, primaryTitle: string, originalTitle: string, isAdult: string, startYear: string, endYear: string, runtimeMinutes: string, genres: string, averageRating: string, numVotes: string, tconst: string, directors: string, writers: string, nconst: string, primaryName: string, birthYear: string, deathYear: string, primaryProfession: string, knownForTitles: string]

In [123]:
big_df.select(big_df.primaryTitle).write.parquet('/user/sandbox/titles2')

In [125]:
big_df.select(big_df.primaryName).write.parquet('/user/sandbox/names2')

## Вывести фильмы с самым высоким рейтингом и их режиссерами

In [ ]:
(title_basics_df.join(ratings_df, 'tconst').
        .filter(ratings_df.numVotes > 10000)
            .filter(title_basics_df.titleType == 'movie')
            .show()
)

# Cache

## Получить ускорение при кэшированнии результата